# Multilingual Toxicity Detection

This notebook covers the full pipeline for binary toxicity classification across English, German, and Finnish.

## Objective
- Build a baseline model: TF-IDF + Logistic Regression.
- Fine-tune multilingual transformer models: XLM-RoBERTa and mBERT.
- Compare all models using the same metrics: F1, Accuracy, AUROC.

## Dataset
- Training split: mainly English.
- Development/Test splits: English, German, Finnish.

## Notes
- No extra external datasets.
<s>- No translation augmentation.</s>

In [ ]:
# TODO: environment and imports
# Expected input: clean Python environment with required packages installed.
# Expected output: all libraries imported, random seeds set, runtime device printed.

# Core
import os
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path

# Visualization (optional in MVP)
import matplotlib.pyplot as plt

# Data preprocessing for baseline
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm
!python -m spacy download fi_core_news_sm
import spacy

# Scikit-learn baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

# Transformer stack
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Reproducibility placeholder
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device placeholder
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 49.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 35.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('

## 3) Data loading and schema check

Expected files (adjust paths as needed):
- `data/train.tsv`
- `data/dev.tsv`
- `data/test.tsv`

Expected columns:
- `text`: raw comment text
- `label`: binary target (0=non-toxic, 1=toxic)
- `language`: language code (`en`, `de`, `fi`) in dev/test (and optionally train if available)

If column names differ, map them in the loading cell below.

In [ ]:
# Running locally; no Google Drive mount needed.

In [ ]:
!pwd

/content


In [ ]:
# TODO: implement data loading
# Expected input: train/dev/test CSV files with text+label (+language for dev/test).
# Expected output: pandas DataFrames train_df, dev_df, test_df with standardized columns.

# Local dataset paths
TRAIN_PATH = "/path/to/project/data/train.tsv"
DEV_PATH = "/path/to/project/data/dev.tsv"
TEST_PATH = "/path/to/project/data/test.tsv"
AUGMENTED_TRAIN_PATH = "/path/to/project/data/train_augmented.tsv"

# TODO: load files
train_df = pd.read_csv(TRAIN_PATH, sep='\t', header=0, quoting=3)
dev_df = pd.read_csv(DEV_PATH, sep='\t', header=0, quoting=3)
test_df = pd.read_csv(TEST_PATH, sep='\t', header=0, quoting=3)
augmented_train_df = pd.read_csv(AUGMENTED_TRAIN_PATH, sep='\t', header=0, quoting=3)

#
# TODO: normalize/rename columns to: text, label, language (where present)
# TODO: enforce label dtype as integer 0/1
dev_df['language'] = dev_df['id'].str[:3]
test_df['language'] = test_df['id'].str[:3]
augmented_train_df['language'] = augmented_train_df['id'].str[:3]

language_map = { 'eng': 'en', 'ger': 'de', 'fin': 'fi' }
dev_df['language'] = dev_df['language'].map(language_map)
test_df['language'] = test_df['language'].map(language_map)
augmented_train_df['language'] = augmented_train_df['language'].map(language_map)


y_train = pd.to_numeric(train_df['label'], errors='coerce').fillna(0).astype(int)
y_dev = pd.to_numeric(dev_df['label'], errors='coerce').fillna(0).astype(int)
y_aug_train = pd.to_numeric(augmented_train_df['label'], errors='coerce').fillna(0).astype(int)

# TODO: sanity checks
# - print split sizes
# - print class distribution per split
# - print language distribution in dev/test
# - check null rows and duplicates
print(len(train_df), len(dev_df), len(test_df))
print(augmented_train_df.head())
print(dev_df.columns)

In [ ]:
print(test_df.columns)

Index(['id', 'text', 'language'], dtype='object')


## 4) Minimal text preprocessing for baseline only

This preprocessing applies only to the TF-IDF + Logistic Regression baseline.

Do **not** apply these normalization steps to BERT inputs.

Planned minimal steps:
- Lowercasing
- Punctuation removal
- Stopword removal
- Stemming **or** lemmatization

**Note!!!**

In the stopword removal phase we only use English stop words? Even with Finnish and German text? Also I think that Wordnet lemmatizer in NLTK is also only for English? [was at least quickly mentioned here ](https://stackoverflow.com/questions/50039310/is-nltk-wordnet-lemmatizer-language-independent)

- Ilkka

In [ ]:
# TODO: implement preprocessing for baseline
# Expected input: train_df/dev_df/test_df with `text` column.
# Expected output: processed text columns for baseline model only.

# TODO: choose one NLP toolkit (e.g., nltk or spacy)
# TODO: create stopword list (language-aware if needed)

nlp_models = {
    'en': spacy.load('en_core_web_sm'),
    'de': spacy.load('de_core_news_sm'),
    'fi': spacy.load('fi_core_news_sm')
}

# def preprocess_for_baseline(text: str) -> str:
#     """Apply minimal deterministic normalization for TF-IDF baseline."""
#     # 1) lowercase
#     # 2) remove punctuation/non-word noise
#     # 3) remove stopwords
#     # 4) stem OR lemmatize
#     # return cleaned_text
#     pass

def preprocess_for_baseline(df, language='language', all_eng=False):

    results = []

    if all_eng:
        nlp = nlp_models['en']
        docs = nlp.pipe(df['text'].astype(str).str.lower(), disable=['ner', 'parser'])
        cleaned = [" ".join([t.lemma_ for t in doc if not t.is_stop and t.is_alpha]) for doc in docs]
        return pd.Series(cleaned, index=df.index)

    else:
        for lang, group in df.groupby(language):
            nlp = nlp_models[lang]
            docs = nlp.pipe(group['text'].astype(str).str.lower(), disable=['ner', 'parser'])
            cleaned = [" ".join([t.lemma_ for t in doc if not t.is_stop and t.is_alpha]) for doc in docs]
            results.append(pd.Series(cleaned, index=group.index))
        return pd.concat(results).sort_index()


# TODO: apply to train/dev/test
train_df['text_baseline'] = preprocess_for_baseline(train_df, all_eng=True)
dev_df['text_baseline'] = preprocess_for_baseline(dev_df)
test_df['text_baseline'] = preprocess_for_baseline(test_df)
augmented_train_df['text_baseline'] = preprocess_for_baseline(augmented_train_df)

# Saving the preprocessed data for convenience
output_dir = "/path/to/project/data"
Path(output_dir).mkdir(parents=True, exist_ok=True)

train_path = os.path.join(output_dir, 'train_baseline.csv')
dev_path = os.path.join(output_dir, 'dev_baseline.csv')
test_path = os.path.join(output_dir, 'test_baseline.csv')
aug_train_path = os.path.join(output_dir, 'train_aug_baseline.csv')

train_df.to_csv(train_path, index=False)
dev_df.to_csv(dev_path, index=False)
test_df.to_csv(test_path, index=False)
augmented_train_df.to_csv(aug_train_path, index=False)

In [ ]:
# Load the preprocessed data if you have run the above cell before
output_dir = "/path/to/project/data"
Path(output_dir).mkdir(parents=True, exist_ok=True)

train_path = os.path.join(output_dir, 'train_baseline.csv')
dev_path = os.path.join(output_dir, 'dev_baseline.csv')
test_path = os.path.join(output_dir, 'test_baseline.csv')
aug_train_path = os.path.join(output_dir, 'train_aug_baseline.csv')

if os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    dev_df = pd.read_csv(dev_path)
    test_df = pd.read_csv(test_path)
    augmented_train_df = pd.read_csv(aug_train_path)

train_df['text_baseline'] = train_df['text_baseline'].fillna("")
dev_df['text_baseline'] = dev_df['text_baseline'].fillna("")
test_df['text_baseline'] = test_df['text_baseline'].fillna("")
augmented_train_df['text_baseline'] = augmented_train_df['text_baseline'].fillna("")

#print(train_df.head)

## 5) Baseline model: TF-IDF + Logistic Regression

This is the interpretable and efficient baseline.

Model flow:
1. Fit TF-IDF on training text.
2. Train Logistic Regression on training labels.
3. Predict labels and probabilities on dev/test.
4. Store outputs in a shared `results` structure.

In [ ]:
# TODO: train TF-IDF + Logistic Regression
# Expected input:
# - train_df/dev_df/test_df with text_baseline and label columns.
# Expected output:
# - baseline predictions and probabilities stored in `results`.

results = {}

# TODO: vectorizer setup
tfidf_vec = TfidfVectorizer(max_features=5000)
tfidf_vec_AUG = TfidfVectorizer(max_features=5000)

# TODO: fit on train and transform all splits
X_train_baseline = tfidf_vec.fit_transform(train_df['text_baseline'])
X_dev_baseline = tfidf_vec.transform(dev_df['text_baseline'])
X_test_baseline = tfidf_vec.transform(test_df['text_baseline'])

X_train_baseline_AUG = tfidf_vec_AUG.fit_transform(augmented_train_df['text_baseline'])
X_dev_baseline_AUG = tfidf_vec_AUG.transform(dev_df['text_baseline'])
X_test_baseline_AUG = tfidf_vec_AUG.transform(test_df['text_baseline'])

# TODO: classifier setup and training
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_baseline, y_train)    # ORIGINAL TRAINING DATA

clf_AUG = LogisticRegression(max_iter=1000)
clf_AUG.fit(X_train_baseline_AUG, y_aug_train)   # AUGMENTED TRAINING DATA

# TODO: predictions
dev_preds_baseline = clf.predict(X_dev_baseline)
test_preds_baseline = clf.predict(X_test_baseline)
dev_probs_baseline = clf.predict_proba(X_dev_baseline)[:, 1]
test_probs_baseline = clf.predict_proba(X_test_baseline)[:, 1]

dev_preds_baseline_AUG = clf_AUG.predict(X_dev_baseline_AUG)
test_preds_baseline_AUG = clf_AUG.predict(X_test_baseline_AUG)
dev_probs_baseline_AUG = clf_AUG.predict_proba(X_dev_baseline_AUG)[:, 1]
test_probs_baseline_AUG = clf_AUG.predict_proba(X_test_baseline_AUG)[:, 1]

# TODO: persist outputs for shared evaluation
results['baseline'] = {
    'dev_preds': dev_preds_baseline,
    'dev_probs': dev_probs_baseline,
    'test_preds': test_preds_baseline,
    'test_probs': test_probs_baseline
}

results['baseline_AUG'] = {
    'dev_preds': dev_preds_baseline_AUG,
    'dev_probs': dev_probs_baseline_AUG,
    'test_preds': test_preds_baseline_AUG,
    'test_probs': test_probs_baseline_AUG
}

# Sanity check
print(f"dev_df accuracy: {(dev_preds_baseline == y_dev).mean():.4f}")
print(f"dev_df accuracy with data augmentation: {(dev_preds_baseline_AUG == y_dev).mean():.4f}")

dev_df accuracy: 0.8781
dev_df accuracy with data augmentation: 0.8776


## 6) Shared evaluation utilities (overall + per-language)

Use one reusable metric pipeline for both models to ensure fair comparison.

Primary metrics (paper-aligned):
- F1-score
- Accuracy
- AUROC

If AUROC is not computable for a subset (for example single-class edge case), report `NaN` and continue.

In [ ]:
# TODO: compute overall and per-language metrics
# Expected input:
# - ground-truth labels in dev_df/test_df
# - prediction arrays + probabilities from `results[model_name]`
# Expected output:
# - reusable metric dictionaries and summary DataFrames

from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

LANGUAGES = ["en", "de", "fi"]

# def compute_binary_metrics(y_true, y_pred, y_proba):
#     """Return F1, accuracy, and AUROC for one subset."""
#     # TODO: compute F1
#     # TODO: compute accuracy
#     # TODO: compute AUROC with try/except fallback to NaN
#     # return {"f1": ..., "accuracy": ..., "auroc": ...}
#     pass

def _positive_class_proba(y_proba):
    """Return 1D positive-class probability vector."""
    arr = np.asarray(y_proba)
    if arr.ndim == 2:
        if arr.shape[1] < 2:
            return arr[:, 0]
        return arr[:, 1]
    return arr

def compute_binary_metrics(y_true, y_pred, y_proba):
    """Return F1, accuracy, and AUROC for one subset."""
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    y_proba_pos = _positive_class_proba(y_proba)
    metrics = {
        "f1": f1_score(y_true_arr, y_pred_arr, average="weighted"),
        "accuracy": accuracy_score(y_true_arr, y_pred_arr),
        "auroc": np.nan,
    }
    try:
        metrics["auroc"] = roc_auc_score(y_true_arr, y_proba_pos)
    except Exception:
        metrics["auroc"] = np.nan
    return metrics

# def evaluate_split_with_languages(df, pred, proba, split_name):
#     """Compute overall + per-language metrics for one split."""
#     # TODO: overall metrics on all rows
#     # TODO: metrics for each language in LANGUAGES
#     # return nested dictionary
#     pass

def evaluate_split_with_languages(df, pred, proba, split_name):
    """Compute overall + per-language metrics for one split."""
    pred_arr = np.asarray(pred)
    proba_arr = np.asarray(proba)
    split_metrics = {
        "split": split_name,
        "overall": compute_binary_metrics(df["label"], pred_arr, proba_arr),
        "by_language": {},
    }
    for lang in LANGUAGES:
        lang_mask = df["language"] == lang
        lang_df = df[lang_mask]
        if len(lang_df) == 0:
            split_metrics["by_language"][lang] = {
                "f1": np.nan,
                "accuracy": np.nan,
                "auroc": np.nan,
            }
            continue
        split_metrics["by_language"][lang] = compute_binary_metrics(
            lang_df["label"],
            pred_arr[lang_mask.to_numpy()],
            proba_arr[lang_mask.to_numpy()],
        )
    return split_metrics


# TODO: helper to flatten nested metric dicts into a table for easy comparison
# def metrics_dict_to_frame(metrics_dict):
#     pass

def metrics_dict_to_frame(metrics_dict):
    """Flatten nested split/language metric dictionaries into a DataFrame."""
    rows = []
    for split_name, split_metrics in metrics_dict.items():
        rows.append(
            {
                "split": split_name,
                "subset": "overall",
                **split_metrics["overall"],
            }
        )
        for lang, lang_metrics in split_metrics["by_language"].items():
            rows.append(
                {
                    "split": split_name,
                    "subset": lang,
                    **lang_metrics,
                }
            )
    return pd.DataFrame(rows)


## 7) Baseline evaluation

Evaluate the baseline model on both dev and test with:
- Overall metrics
- Per-language metrics (`en`, `de`, `fi`)

Store all outputs under `results["tfidf_logreg"]["metrics"]` for direct comparison later.

In [ ]:
# evaluate TF-IDF + Logistic Regression
# Expected input:
# - dev_df/test_df with true labels and languages
# - results["baseline"] predictions/probabilities
# Expected output:
# - metric dictionaries for dev and test stored under baseline result key

baseline_dev_metrics = evaluate_split_with_languages(
    dev_df,
    results["baseline"]["dev_preds"],
    results["baseline"]["dev_probs"],
    split_name="dev",
)

results["baseline"]["metrics"] = {
    "dev": baseline_dev_metrics
}

# display quick baseline metric tables
baseline_metrics_df = metrics_dict_to_frame(results["baseline"]["metrics"])

print("=== BASELINE OVERALL METRICS ===")
print(baseline_metrics_df[baseline_metrics_df["subset"] == "overall"].reset_index(drop=True))

print("\n=== BASELINE PER-LANGUAGE METRICS ===")
print(baseline_metrics_df[baseline_metrics_df["subset"] != "overall"].reset_index(drop=True))


# EVALUATING THE BASELINE MODEL TRAINED ON THE AUGMENTED DATA:
baseline_AUG_dev_metrics = evaluate_split_with_languages(
    dev_df,
    results["baseline_AUG"]["dev_preds"],
    results["baseline_AUG"]["dev_probs"],
    split_name="dev",
)

results["baseline_AUG"]["metrics"] = {
    "dev": baseline_AUG_dev_metrics
}

# display quick baseline metric tables
baseline_metrics_AUG_df = metrics_dict_to_frame(results["baseline_AUG"]["metrics"])

print("=== AUGMENTED BASELINE OVERALL METRICS ===")
print(baseline_metrics_AUG_df[baseline_metrics_df["subset"] == "overall"].reset_index(drop=True))

print("\n=== AUGMENTED BASELINE PER-LANGUAGE METRICS ===")
print(baseline_metrics_AUG_df[baseline_metrics_df["subset"] != "overall"].reset_index(drop=True))

=== BASELINE OVERALL METRICS ===
  split   subset        f1  accuracy     auroc
0   dev  overall  0.874506  0.878106  0.919017

=== BASELINE PER-LANGUAGE METRICS ===
  split subset        f1  accuracy     auroc
0   dev     en  0.912613  0.913545  0.967048
1   dev     de  0.674670  0.745500  0.584852
2   dev     fi  0.118798  0.255000  0.546333
=== AUGMENTED BASELINE OVERALL METRICS ===
  split   subset        f1  accuracy     auroc
0   dev  overall  0.874889  0.877576  0.936674

=== AUGMENTED BASELINE PER-LANGUAGE METRICS ===
  split subset        f1  accuracy     auroc
0   dev     en  0.910110  0.911091  0.965921
1   dev     de  0.710071  0.741500  0.654061
2   dev     fi  0.376476  0.395000  0.594933


## 8) Transformer fine-tuning

Two checkpoints are trained:
- Option A: `bert-base-multilingual-cased`
- Option B: `xlm-roberta-base`

Hyperparameters: 2 epochs, lr=2e-5, batch size 32.

In [ ]:
# TODO: fine-tune multilingual BERT
# Expected input:
# - train/dev/test DataFrames with raw text and labels
# Expected output:
# - fine-tuned model predictions/probabilities saved in `results["multilingual_bert"]`

MODEL_CHECKPOINT = "xlm-roberta-base" #"bert-base-multilingual-cased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
NUM_EPOCHS = 2

# TODO: convert DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.rename_column("label", "labels")
dev_dataset = dev_dataset.rename_column("label", "labels")
# TODO: tokenizer + tokenization function
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


# TODO: map tokenization over datasets and set torch format
train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
dev_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
# TODO: model initialization
num_labels = train_df["label"].nunique()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels
)
# TODO: training arguments and trainer
training_args = TrainingArguments(
    output_dir="/path/to/project/data/xlm_results",

    # Learning
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,

    # Batch
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,

    # Speed
    fp16=True,
    dataloader_num_workers=2,

    # Logging
    logging_dir="./logs",
    logging_steps=200,

    # Saving
    save_strategy="epoch",
    save_total_limit=1,

    # Eval
    do_eval=False,

    # Misc
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# TODO: run training
trainer.train()

In [ ]:
trainer.save_model("/path/to/project/data/xlm_final")
tokenizer.save_pretrained("/path/to/project/data/xlm_final")

ROBERTA with Augmented data

In [ ]:
# TODO: fine-tune multilingual BERT
# Expected input:
# - train/dev/test DataFrames with raw text and labels
# Expected output:
# - fine-tuned model predictions/probabilities saved in `results["multilingual_bert"]`

MODEL_CHECKPOINT = "xlm-roberta-base" #"bert-base-multilingual-cased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
NUM_EPOCHS = 2

# TODO: convert DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(augmented_train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.rename_column("label", "labels")
dev_dataset = dev_dataset.rename_column("label", "labels")
# TODO: tokenizer + tokenization function
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


# TODO: map tokenization over datasets and set torch format
train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
dev_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
# TODO: model initialization
num_labels = train_df["label"].nunique()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels
)
# TODO: training arguments and trainer
training_args = TrainingArguments(
    output_dir="/path/to/project/data/xlm_results",

    # Learning
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,

    # Batch
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,

    # Speed
    fp16=True,
    dataloader_num_workers=2,

    # Logging
    logging_dir="./logs",
    logging_steps=200,

    # Saving
    save_strategy="epoch",
    save_total_limit=1,

    # Eval
    do_eval=False,

    # Misc
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# TODO: run training
trainer.train()

In [ ]:
trainer.save_model("/path/to/project/data/xlm_final_augmented")
tokenizer.save_pretrained("/path/to/project/data/xlm_final_augmented")

### Option B: bert-base-multilingual-cased

Let's now try training the other model option as well!

In [ ]:
MODEL_CHECKPOINT = "bert-base-multilingual-cased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
NUM_EPOCHS = 2

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.rename_column("label", "labels")
dev_dataset = dev_dataset.rename_column("label", "labels")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
dev_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

num_labels = train_df["label"].nunique()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
)

training_args = TrainingArguments(
    output_dir="/path/to/project/data/bert_results",
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    fp16=True,
    dataloader_num_workers=2,
    logging_dir="./logs",
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=2,
    do_eval=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()
trainer.save_model("/path/to/project/data/bert_final")
tokenizer.save_pretrained("/path/to/project/data/bert_final")

## 9) Transformer evaluation

Evaluate multilingual BERT with the exact same metric helpers used by baseline:
- Overall dev/test metrics
- Per-language dev/test metrics (`en`, `de`, `fi`)

This keeps the comparison direct and fair.

In [ ]:
# Updated so that
# Tokenization for evaluation is handled inside the section-9 helper
# that loads each saved model and its tokenizer from disk.
#test_dataset = test_dataset.map(tokenize, batched=True)
#test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [ ]:
# evaluate both saved transformer models (XLM-R + bonus BERT)
# Expected input:
# - saved model/tokenizer directories
# - dev_df labels and language columns
# - test_df text (no labels required)
# - shared utilities from section 6
# Expected output:
# - predictions/probabilities for dev+test and metrics on dev only

XLM_MODEL_PATH = "/path/to/project/data/xlm_final"
XLM_AUG_MODEL_PATH = "/path/to/project/data/xlm_final_augmented"
BERT_MODEL_PATH = "/path/to/project/data/bert_final"

if "results" not in globals():
    results = {}


def process_outputs(outputs):
    logits = outputs.predictions
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    return probs, preds


def build_eval_dataset(df, tokenizer, max_length=128, include_labels=False):
    if include_labels:
        dataset = Dataset.from_pandas(df[["text", "label"]].copy())
        dataset = dataset.rename_column("label", "labels")
    else:
        dataset = Dataset.from_pandas(df[["text"]].copy())

    def _tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    dataset = dataset.map(_tokenize, batched=True)
    columns = ["input_ids", "attention_mask"]
    if include_labels:
        columns.append("labels")
    if "token_type_ids" in dataset.column_names:
        columns.append("token_type_ids")
    dataset.set_format(type="torch", columns=columns)
    return dataset


def evaluate_saved_model(model_path, model_key):
    tokenizer_local = AutoTokenizer.from_pretrained(model_path)
    model_local = AutoModelForSequenceClassification.from_pretrained(model_path)

    dev_eval_dataset = build_eval_dataset(dev_df, tokenizer_local, include_labels=True)
    test_eval_dataset = build_eval_dataset(test_df, tokenizer_local, include_labels=False)

    eval_trainer = Trainer(model=model_local)
    dev_outputs = eval_trainer.predict(dev_eval_dataset)
    test_outputs = eval_trainer.predict(test_eval_dataset)

    dev_probs, dev_preds = process_outputs(dev_outputs)
    test_probs, test_preds = process_outputs(test_outputs)

    results[model_key] = {
        "dev_probs": dev_probs,
        "dev_preds": dev_preds,
        "test_probs": test_probs,
        "test_preds": test_preds,
        "metrics": {
            "dev": evaluate_split_with_languages(dev_df, dev_preds, dev_probs, split_name="dev"),
        },
    }

    metrics_df = metrics_dict_to_frame(results[model_key]["metrics"])
    print(f"\n=== {model_key.upper()} OVERALL METRICS (DEV) ===")
    print(metrics_df[metrics_df["subset"] == "overall"].reset_index(drop=True))
    print(f"\n=== {model_key.upper()} PER-LANGUAGE METRICS (DEV) ===")
    print(metrics_df[metrics_df["subset"] != "overall"].reset_index(drop=True))


evaluate_saved_model(XLM_MODEL_PATH, "roberta")
evaluate_saved_model(XLM_AUG_MODEL_PATH, "roberta_aug")
evaluate_saved_model(BERT_MODEL_PATH, "bert_base_multilingual_cased")

In [ ]:
for model_name in ["roberta", "roberta_aug", "bert_base_multilingual_cased"]:
    if model_name not in results or "test_preds" not in results[model_name]:
        raise ValueError(f"Missing test predictions for model '{model_name}'. Run section 9 first.")

    df_out = pd.DataFrame({
        "id": test_df["id"],
        "predicted": results[model_name]["test_preds"],
    })

    out_path = f"/path/to/project/data/xlm_results/test_predictions_{model_name}.tsv"
    df_out.to_csv(out_path, sep="\t", index=False)
    print(f"Saved: {out_path}")

## 10) Side-by-side comparison table

- One compact overall table comparing all models.
- One compact per-language table (EN/DE/FI).

In [ ]:
# build model comparison tables
# Expected input:
# - results["baseline"]["metrics"]
# - results["roberta"]["metrics"]
# - results["bert_base_multilingual_cased"]["metrics"]
# Expected output:
# - overall comparison DataFrame
# - per-language comparison DataFrame

required_models = ["baseline", "roberta","roberta_aug", "bert_base_multilingual_cased"]
for model_name in required_models:
    if model_name not in results or "metrics" not in results[model_name]:
        raise ValueError(f"Missing metrics for model '{model_name}'. Run evaluation cells first.")

# flatten each model's dev/test metrics into tabular rows
model_frames = []
for model_name in required_models:
    model_df = metrics_dict_to_frame(results[model_name]["metrics"]).copy()
    model_df["model"] = model_name
    model_frames.append(model_df)

# combine and print/save table
comparison_df = pd.concat(model_frames, ignore_index=True)
comparison_df = comparison_df[["model", "split", "subset", "f1", "accuracy", "auroc"]]

overall_comparison_df = comparison_df[comparison_df["subset"] == "overall"].reset_index(drop=True)
per_language_comparison_df = comparison_df[comparison_df["subset"] != "overall"].reset_index(drop=True)

print("=== OVERALL COMPARISON ===")
print(overall_comparison_df)

print("\n=== PER-LANGUAGE COMPARISON ===")
print(per_language_comparison_df)

# optional split tables
overall_comparison_df.to_csv("overall_comparison.csv", index=False)
per_language_comparison_df.to_csv("per_language_comparison.csv", index=False)

=== OVERALL COMPARISON ===
                          model split   subset        f1  accuracy     auroc
0                      baseline   dev  overall  0.870564  0.874394  0.918318
1                       roberta   dev  overall  0.914968  0.914773  0.967811
2  bert_base_multilingual_cased   dev  overall  0.906465  0.906818  0.964930

=== PER-LANGUAGE COMPARISON ===
                          model split subset        f1  accuracy     auroc
0                      baseline   dev     en  0.908995  0.910091  0.963878
1                      baseline   dev     de  0.672786  0.740500  0.562124
2                      baseline   dev     fi  0.117041  0.250000  0.498333
3                       roberta   dev     en  0.942301  0.942000  0.984828
4                       roberta   dev     de  0.780246  0.785000  0.799212
5                       roberta   dev     fi  0.733878  0.715000  0.834400
6  bert_base_multilingual_cased   dev     en  0.939560  0.939273  0.983361
7  bert_base_multilingual_cased 

## 11) Conclusions (to be filled after running)

1. Which model performs best overall (F1/Accuracy/AUROC)?
2. Is there a cross-lingual performance drop from English to German/Finnish?
3. What is the next experiment to run?